In [1]:
import time
import numpy as np
import sys
import pandas as pd
# import nrm
import csv
import subprocess
import os
import tarfile
import random
from datetime import datetime
import torch

In [2]:
def normalize(data, MIN, MAX):
    return np.round((np.float64(data) - MIN) / (MAX - MIN), decimals=4)

class FCNetwork(torch.nn.Module):
  def __init__(self, layers=[20,20]):
    super(FCNetwork, self).__init__()
    # self.all_observations = torch.tensor(stack_observations(env), dtype=torch.float32)
    dim_input = 7
    dim_output = 32
    net_layers = []

    dim = dim_input
    for i, layer_size in enumerate(layers):
      net_layers.append(torch.nn.Linear(dim, layer_size))
      net_layers.append(torch.nn.ReLU())
      dim = layer_size
    net_layers.append(torch.nn.Linear(dim, dim_output))
    self.layers = net_layers
    self.network = torch.nn.Sequential(*net_layers)

  def forward(self, states):
    # observations = torch.index_select(self.all_observations, 0, states)
    states_tensor = torch.tensor(states, dtype=torch.float32)  # Ensure the correct dtype
    return self.network(states_tensor)

  def print_weights(self):
    for name, param in self.named_parameters():
        if param.requires_grad:
            print(f"{name}: {param.data.numpy()}")
            
# model = FCNetwork(layers=[5,5])

i = 0

policy_folder = '/home/cc/summer2024/main_codes/'  # Default policy file
# policy_file = os.path.join(policy_folder,'BCQ_SYS_0_20240929_183736.pt')
while i < len(sys.argv):
    if sys.argv[i] == '--application':
        APPLICATION = sys.argv[i+1]
        i += 1
    elif sys.argv[i] == '--policy':
        policy_name = sys.argv[i+1]  # Update policy file from argument
        policy_file = os.path.join(policy_folder, policy_name)
        i += 1
    i +=1

In [3]:
def get_data_dir(subfolder):
    current_dir = os.getcwd()
    print(current_dir)
    return os.path.join(current_dir, "experiment_data", f"{subfolder}")

DATA_DIR = get_data_dir("training_data")

csv_file_path = f'{DATA_DIR}/training_dataset.csv'


/Users/akhileshraj/Documents/summer2024/main_codes


In [4]:
model = FCNetwork(layers=[20, 20])
policy_name = "trained_network_weights_20260212_111852_all_preference_model_based_0.01_0.001.pth"
policy_file = os.path.join("/Users/akhileshraj/Documents/summer2024/main_codes/trained_models", policy_name)
model.load_state_dict(torch.load(policy_file))
model.eval()

/var/folders/6w/z0z_k2497g9dpnm_80x2647h0000gn/T/ipykernel_40533/3839573743.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(policy_file)

FCNetwork(
  (network): Sequential(
    (0): Linear(in_features=7, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=32, bias=True)
  )
)

In [9]:
data = pd.read_csv(csv_file_path)

df = pd.DataFrame(data)
ACTIONS = [78.0, 83.0, 89.0, 95.0, 101.0, 107.0, 112.0, 118.0, 124.0, 130.0, 136.0, 141.0, 147.0, 153.0, 159.0, 165.0]
pow_pref = 0.9
prog_pref = 1-pow_pref
preference = np.array([pow_pref,prog_pref], dtype=np.float32)   # (2,)
application = "ones"
for i in range(len(df)):
    if application in df.iloc[i][0]:
 
        state = np.array(df.iloc[i][1:6], dtype=np.float32)   # (5,)

        # model input: concatenate -> (7,), then add batch dim -> (1,7)
        s_vecs = np.concatenate([state, preference], axis=0)
        s_vecs_t = torch.from_numpy(s_vecs).unsqueeze(0)      # (1, 7)

        # model forward -> suppose it returns (1, 32)
        suggested_action = model(s_vecs_t)

        # reshape to actions × objectives = (1, 16, 2)
        act_vec = suggested_action.view(1, 16, 2)             # (B=1, A=16, L=2)

        # preference for bmm: make it (B=1, 1, L=2)
        pref_t = torch.from_numpy(preference).view(1, 1, 2)   # (1, 1, 2)

        # for bmm we need (B, 1, 2) @ (B, 2, 16) -> (B, 1, 16)
        q_for_bmm = act_vec.transpose(1, 2)                   # (1, 2, 16)

        # scalarized Q over objectives per action
        # scalarized = torch.bmm(pref_t, q_for_bmm).squeeze(0).squeeze(0)  # (16,
        scalarized = q_for_bmm[0,0,:]/q_for_bmm[0,1,:]
        argmax = (np.argmax(scalarized.detach().numpy(),axis=-1))
        # print(suggested_action,"\n",argmax,ACTIONS[argmax])
        print(f"{i+1},.....,{df.iloc[i][0]}.....{ACTIONS[argmax]}")

1,.....,ones-stream-add.....78.0
2,.....,ones-stream-add.....124.0
3,.....,ones-stream-add.....83.0
4,.....,ones-stream-add.....83.0
5,.....,ones-stream-add.....124.0
6,.....,ones-stream-add.....83.0
7,.....,ones-stream-add.....78.0
8,.....,ones-stream-add.....124.0
9,.....,ones-stream-add.....124.0
10,.....,ones-stream-add.....124.0
11,.....,ones-stream-add.....83.0
12,.....,ones-stream-add.....83.0
13,.....,ones-stream-add.....78.0
14,.....,ones-stream-add.....83.0
15,.....,ones-stream-add.....83.0
16,.....,ones-stream-add.....83.0
17,.....,ones-stream-add.....124.0
18,.....,ones-stream-add.....83.0
19,.....,ones-stream-add.....78.0
20,.....,ones-stream-add.....83.0
21,.....,ones-stream-add.....124.0
22,.....,ones-stream-add.....83.0
23,.....,ones-stream-add.....124.0
24,.....,ones-stream-add.....83.0
25,.....,ones-stream-add.....78.0
26,.....,ones-stream-add.....83.0
27,.....,ones-stream-add.....124.0
28,.....,ones-stream-add.....124.0
29,.....,ones-stream-add.....83.0
30,.....,ones

/var/folders/6w/z0z_k2497g9dpnm_80x2647h0000gn/T/ipykernel_40533/497899282.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if application in df.iloc[i][0]:
/var/folders/6w/z0z_k2497g9dpnm_80x2647h0000gn/T/ipykernel_40533/3890130527.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  states_tensor = torch.tensor(states, dtype=torch.float32)  # Ensure the correct dtype
/var/folders/6w/z0z_k2497g9dpnm_80x2647h0000gn/T/ipykernel_40533/497899282.py:35: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by 